# 02 — Diana: Stamp & Signature Detection

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Trains a **2-class** detector (`stamp`, `signature`) on real SignverOD + StaVer data, evaluates per class on a real held-out split, then runs inference on the invoice images.

| | |
|---|---|
| **Inputs** | Drive: `datasets/signatures/`, `datasets/stamps/`, invoice manifest |
| **Outputs** | `stamp_signature_predictions.csv`, metrics JSON, figure, weights |
| **Expected runtime** | ~20–40 min on a T4 (budget-optimized: imgsz 640 / ≤50 epochs — see the training cell; checkpoints go to Drive so a disconnect is resumable) |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/diana/<kind>/` — the *latest* copy
- `runs/diana/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### The rule that cannot bend
`stamp` and `signature` are **always two separate labels**. Never merge them into one
"authorization mark" class, never rename either string — the final JSON schema, the Streamlit UI,
and the Pistac.io readiness logic all assume both exist independently.

### Two datasets, two shapes of annotation
- **SignverOD** — `train.csv`/`test.csv` give `bbox` as a *stringified JSON list of **normalized**
  `[xmin, ymin, w, h]`*, joined to `image_ids.csv` for pixel dimensions. Categories are
  `1=signature, 2=initials, 3=redaction, 4=date` → **only category 1** is our `signature`.
- **StaVer** — ships **no boxes at all**, only binary ground-truth *masks*. Boxes have to be
  derived with connected components, cross-checked against `numStamps` in the info files.

### Domain gap — be honest about it
Neither source is an invoice. Metrics below are computed on a **held-out split of the real source
data** (legitimate, real numbers). Inference on invoices has **no ground truth**, so report
detection counts and confidence distributions there — never a precision/recall figure.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("ultralytics", "opencv-python-headless", "pandas")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Datasets: read straight from Google Drive (NO Kaggle token needed) ------
# Both were pre-downloaded into Drive.
# Paths are resolved tolerantly: if a dataset was copied one level too deep
# (e.g. ocr_multitype/invoice/train/... instead of ocr_multitype/train/...),
# it is found anyway and a NOTE is printed. No re-upload needed.
DATA = paths.inputs / "datasets"

SIG = CB.resolve_dataset_root(DATA / "signatures", ['images', 'image_ids.csv'])
STA = CB.resolve_dataset_root(DATA / "stamps", ['scans', 'ground-truth-maps'])

print(f"  signatures     -> {SIG}")
print(f"                    " f"{sum(1 for _ in SIG.rglob(chr(42)) if _.is_file()):,} files")
print(f"  stamps         -> {STA}")
print(f"                    " f"{sum(1 for _ in STA.rglob(chr(42)) if _.is_file()):,} files")

In [ ]:
# --- SignverOD -> pixel boxes for category_id == 1 (signature) ----------------
import pandas as pd, ast, cv2, numpy as np

ids = pd.read_csv(SIG / "image_ids.csv")            # height,width,id,file_name
tr  = pd.read_csv(SIG / "train.csv")                # area,bbox,category_id,id,image_id

df = tr[tr.category_id == 1].merge(ids, left_on="image_id", right_on="id", suffixes=("", "_img"))
print("signature annotations:", len(df), "| images:", df.file_name.nunique())

def to_px(row):
    x, y, w, h = ast.literal_eval(row["bbox"])       # normalized [xmin,ymin,w,h]
    W, H = row["width"], row["height"]
    return pd.Series([x * W, y * H, (x + w) * W, (y + h) * H])

df[["xmin", "ymin", "xmax", "ymax"]] = df.apply(to_px, axis=1)
# sanity: normalized area should reconstruct
print("area check:", np.allclose(df.area.head(20),
      ((df.xmax-df.xmin)/df.width * (df.ymax-df.ymin)/df.height).head(20), atol=2e-3))
df.head(3)[["file_name", "xmin", "ymin", "xmax", "ymax"]]

In [ ]:
# --- StaVer -> boxes from the binary GT masks --------------------------------
# resolve_files_dir guards against StaVer's scans/scans double-nesting surviving the copy
SCAN_DIR = CB.resolve_files_dir(STA / "scans", "*.png")
MASK_DIR = CB.resolve_files_dir(STA / "ground-truth-maps", "*.png")
INFO_DIR = CB.resolve_files_dir(STA / "info", "*.txt")
scans = {p.stem: p for p in SCAN_DIR.glob("*.png")}
masks = sorted(MASK_DIR.glob("*-gt.png"))
print("scans:", len(scans), "| gt masks:", len(masks))

def boxes_from_mask(mask_path, min_area_frac=2e-4):
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    # stamps are the dark/coloured ink on a white GT map - binarise either polarity
    bw = (m < 128).astype(np.uint8)
    if bw.mean() > .5:
        bw = 1 - bw
    n, _, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
    H, W = m.shape
    out = []
    for i in range(1, n):
        x, y, w, h, a = stats[i]
        if a >= min_area_frac * H * W:
            out.append((x, y, x + w, y + h))
    return out, (W, H)

stamp_rows = []
for mp in masks:
    stem = mp.stem.replace("-gt", "")
    if stem not in scans:
        continue
    bx, (W, H) = boxes_from_mask(mp)
    info = INFO_DIR / f"{stem}.txt"
    expected = None
    if info.exists():
        try:
            expected = int(info.read_text().strip().splitlines()[-1].split()[2])
        except Exception:
            pass
    for b in bx:
        stamp_rows.append({"file": scans[stem], "stem": stem, "W": W, "H": H,
                           "xmin": b[0], "ymin": b[1], "xmax": b[2], "ymax": b[3],
                           "n_found": len(bx), "n_expected": expected})
sdf = pd.DataFrame(stamp_rows)
print("stamp boxes:", len(sdf), "| images:", sdf.stem.nunique())
ok = sdf.drop_duplicates("stem").dropna(subset=["n_expected"])
print("images where found == expected numStamps: %.1f%%" %
      (100 * (ok.n_found == ok.n_expected).mean()))

In [ ]:
# --- Build a 2-class YOLO dataset --------------------------------------------
D = Path("/content/yolo")
for sp in ["train", "val"]:
    (D / sp / "images").mkdir(parents=True, exist_ok=True)
    (D / sp / "labels").mkdir(parents=True, exist_ok=True)

NAMES = ["stamp", "signature"]          # index 0 = stamp, 1 = signature - do not reorder

def write(split, img_path, boxes, cls_idx, size=None):
    dst = D / split / "images" / f"{cls_idx}_{Path(img_path).stem}.jpg"
    im = cv2.imread(str(img_path))
    if im is None:
        return False
    H, W = im.shape[:2]
    cv2.imwrite(str(dst), im)
    lines = []
    for (x1, y1, x2, y2) in boxes:
        cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
        bw, bh = (x2 - x1) / W, (y2 - y1) / H
        if bw <= 0 or bh <= 0:
            continue
        lines.append(f"{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    (D / split / "labels" / f"{dst.stem}.txt").write_text("\n".join(lines))
    return True

cap = P.get("max_images_per_class") or 10**9

# stamps
stems = sorted(sdf.stem.unique())[:cap]
for i, s in enumerate(stems):
    g = sdf[sdf.stem == s]
    write("val" if i % 6 == 0 else "train", g.file.iloc[0],
          g[["xmin", "ymin", "xmax", "ymax"]].values, 0)

# signatures
sig_imgs = sorted(df.file_name.unique())[:cap]
IMG_DIR = CB.resolve_files_dir(SIG / "images", "*.png")
for i, fn in enumerate(sig_imgs):
    g = df[df.file_name == fn]
    p = IMG_DIR / fn
    if not p.exists():
        continue
    write("val" if i % 6 == 0 else "train", p, g[["xmin", "ymin", "xmax", "ymax"]].values, 1)

yaml = D / "data.yaml"
yaml.write_text(f"path: {D}\ntrain: train/images\nval: val/images\n"
                f"nc: 2\nnames: {NAMES}\n")
print("train:", len(list((D/'train'/'images').glob('*'))),
      "| val:", len(list((D/'val'/'images').glob('*'))))

In [ ]:
# --- Train: FINAL attempt — imgsz 768, YOLO default aug (NO rotation/shear) ----
# Evidence: best IoU (0.82) came from YOLO's DEFAULT augmentation; attempt 2 added
# degrees=8/shear=2 and IoU dropped to ~0.80. This run keeps the resolution bump but
# removes the harmful geometric distortion (reverts everything else to defaults).
# yolov8n stays (yolov8s at 768 would need ~9h, far over the 4h budget).
from ultralytics import YOLO

RUN_NAME = "stamp_sig_768_defaug"   # fresh run => won't resume attempt 2's weights

P["imgsz"]  = 768
P["epochs"] = 75

CKPT = root / "outputs" / "diana" / "runs"       # on DRIVE -- survives a runtime recycle
CKPT.mkdir(parents=True, exist_ok=True)
LAST = CKPT / RUN_NAME / "weights" / "last.pt"

if LAST.exists():
    # resumes THIS run only if it disconnected mid-way (re-run dataset cells first, no GPU cost)
    print("resuming from", LAST)
    model = YOLO(str(LAST))
    model.train(resume=True)
else:
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(yaml), epochs=P["epochs"], imgsz=P["imgsz"], batch=P["batch"],
        workers=P.get("workers", 2), device=0, patience=P.get("patience", 20),
        project=str(CKPT), name=RUN_NAME, exist_ok=True,
        save_period=5, verbose=True,
        degrees=0.0, shear=0.0,     # kill rotation/shear; keep YOLO's default (good) aug
    )
BEST = CKPT / RUN_NAME / "weights" / "best.pt"
print("best weights:", BEST, BEST.exists(), "| imgsz", P["imgsz"], "epochs", P["epochs"])


In [ ]:
# --- Per-class precision / recall / mean IoU on the REAL held-out split -------
# Sweeps a few confidence thresholds so you can pick the best recall/precision balance
# (signature recall was the weak spot). The canonical metrics used downstream are set
# at the bottom -- change CONF there if the sweep tells you to.
from src.iou import compute_iou           # Jordan's module - import, never reimplement

val_imgs = sorted((D / "val" / "images").glob("*.jpg"))
m = YOLO(str(BEST))

def eval_at(conf):
    stats = {n: {"tp": 0, "fp": 0, "fn": 0, "ious": []} for n in NAMES}
    for ip in val_imgs:
        lab = D / "val" / "labels" / f"{ip.stem}.txt"
        im = cv2.imread(str(ip)); H, W = im.shape[:2]
        gt = []
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            ci, cx, cy, bw, bh = line.split()
            ci = int(ci); cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
            gt.append((ci, [(cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H]))
        pr = m.predict(str(ip), conf=conf, verbose=False)[0]
        preds = [(int(c), b.tolist()) for c, b in
                 zip(pr.boxes.cls.cpu().numpy(), pr.boxes.xyxy.cpu().numpy())]
        for ci, name in enumerate(NAMES):
            g = [b for k, b in gt if k == ci]
            p_ = [b for k, b in preds if k == ci]
            used = set()
            for pb in p_:
                best, bi = 0.0, -1
                for j, gb in enumerate(g):
                    if j in used:
                        continue
                    v = compute_iou(pb, gb)
                    if v > best:
                        best, bi = v, j
                if best >= 0.5:
                    stats[name]["tp"] += 1; stats[name]["ious"].append(best); used.add(bi)
                else:
                    stats[name]["fp"] += 1
            stats[name]["fn"] += len(g) - len(used)
    out = {}
    for n, s in stats.items():
        tp, fp, fn = s["tp"], s["fp"], s["fn"]
        out[n] = {
            "precision": round(tp / (tp + fp), 4) if tp + fp else 0.0,
            "recall":    round(tp / (tp + fn), 4) if tp + fn else 0.0,
            "mean_iou":  round(float(np.mean(s["ious"])), 4) if s["ious"] else 0.0,
            "tp": tp, "fp": fp, "fn": fn,
        }
    return out

# --- Sweep: compare thresholds (informational; does not affect downstream) ----
print("conf | class     |  prec   recall  mIoU    tp  fp  fn")
print("-" * 56)
for c in (0.15, 0.25, 0.40):
    r = eval_at(c)
    for n in NAMES:
        d = r[n]
        print(f"{c:.2f} | {n:9s} | {d['precision']:.3f}  {d['recall']:.3f}  {d['mean_iou']:.3f}  "
              f"{d['tp']:3d} {d['fp']:3d} {d['fn']:3d}")

# --- Canonical metrics used by the rest of the notebook -----------------------
# If the sweep shows a lower conf gives clearly better signature recall with acceptable
# precision, change CONF here (e.g. 0.15) and re-run this cell + cells 12 and 14.
CONF = 0.25
metrics = eval_at(CONF)
print("\nCanonical (CONF =", CONF, "):")
print(json.dumps(metrics, indent=2))


In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "diana",
    }
    b.update(kw)
    return b


In [ ]:
# --- Inference on the REAL invoices (no GT here - counts only) ----------------
man = pd.read_csv(paths.inputs / "invoice_manifest.csv")
rows = []
for r in man.itertuples():
    ip = root / r.image_path
    if not ip.exists():
        continue
    pr = m.predict(str(ip), conf=CONF, verbose=False)[0]
    for cls, box, cf in zip(pr.boxes.cls.cpu().numpy(),
                            pr.boxes.xyxy.cpu().numpy(),
                            pr.boxes.conf.cpu().numpy()):
        rows.append({"document_id": r.document_id, "image_path": r.image_path,
                     "label": NAMES[int(cls)],
                     "xmin": float(box[0]), "ymin": float(box[1]),
                     "xmax": float(box[2]), "ymax": float(box[3]),
                     "confidence": round(float(cf), 4)})

pred = pd.DataFrame(rows, columns=["document_id", "image_path", "label",
                                   "xmin", "ymin", "xmax", "ymax", "confidence"])
assert pred.empty or set(pred.label) <= {"stamp", "signature"}, "label vocabulary violated!"
OUTD = Path("/content/out"); OUTD.mkdir(exist_ok=True)
pred.to_csv(OUTD / "stamp_signature_predictions.csv", index=False)

print("invoices with a detection:", pred.document_id.nunique(), "/", len(man))
print(pred.label.value_counts().to_dict())
print(pred.groupby("label").confidence.describe()[["mean", "min", "max"]] if not pred.empty else "")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Figure + weights ---------------------------------------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

FIG = Path("/content/out/figures"); FIG.mkdir(parents=True, exist_ok=True)
show = (pred.document_id.drop_duplicates().head(6).tolist() if not pred.empty
        else man.document_id.head(6).tolist())
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for a, d in zip(ax.ravel(), show):
    r = man[man.document_id == d].iloc[0]
    a.imshow(plt.imread(root / r.image_path)); a.axis("off"); a.set_title(d, fontsize=9)
    for q in pred[pred.document_id == d].itertuples():
        col = "tab:red" if q.label == "stamp" else "tab:blue"
        a.add_patch(Rectangle((q.xmin, q.ymin), q.xmax-q.xmin, q.ymax-q.ymin,
                              fill=False, lw=2, ec=col))
        a.text(q.xmin, q.ymin-4, f"{q.label} {q.confidence:.2f}", color=col, fontsize=7)
fig.suptitle("Stamp (red) / signature (blue) detections on real invoices", fontsize=13)
fig.tight_layout(); fig.savefig(FIG / "stamp_signature_detection_examples.png", dpi=150)
plt.close(fig)

MD = Path("/content/out/models"); (MD/"stamp_detector").mkdir(parents=True, exist_ok=True)
(MD/"signature_detector").mkdir(parents=True, exist_ok=True)
for sub in ["stamp_detector", "signature_detector"]:
    shutil.copyfile(BEST, MD / sub / "best.pt")
    (MD / sub / "README.md").write_text(
        f"# {sub}\n\nSingle YOLOv8n **2-class** model (`stamp`, `signature`); the same weights "
        f"are stored under both folder names to satisfy the output contract.\n\n"
        f"Trained on real SignverOD (category_id==1) + StaVer (boxes derived from GT masks).\n"
        f"Profile `colab_gpu`: epochs={P['epochs']}, imgsz={P['imgsz']}, batch={P['batch']}.\n\n"
        f"Metrics: {json.dumps(metrics)}\n", encoding="utf-8")

met = Path("/content/out/stamp_signature_metrics.json")
payload = dict(metrics)
payload["_run"] = run_block(model="yolov8n",
                            n_train_images=len(list((D/'train'/'images').glob('*'))),
                            eval_set="real held-out split of SignverOD+StaVer",
                            conf_threshold=CONF, iou_match_threshold=0.5)
payload["_invoice_inference"] = {
    "note": "No ground truth on invoices - counts only, never precision/recall.",
    "invoices_scored": int(len(man)),
    "invoices_with_detection": int(pred.document_id.nunique()) if not pred.empty else 0,
    "detections_by_label": pred.label.value_counts().to_dict() if not pred.empty else {},
}
met.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps(payload, indent=2)[:900])

In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
#
to_publish = [
    ("predictions", OUTD / "stamp_signature_predictions.csv"),
    ("metrics", met),
    ("figures", FIG / "stamp_signature_detection_examples.png"),
    ("models", MD),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("diana", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("diana"))
print("Archive ->", paths.run_dir("diana", timestamp=RUN_TS))

In [ ]:
# --- Hand off to Damir + Hessam ----------------------------------------------
up = paths.inputs / "upstream" / "diana"; up.mkdir(parents=True, exist_ok=True)
shutil.copyfile(OUTD / "stamp_signature_predictions.csv",
                up / "stamp_signature_predictions.csv")
print("handed off ->", up)

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/diana_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. One 2-class model vs two separate detectors — what you chose and why.
2. Deriving StaVer boxes from masks: what the connected-component filter got wrong, and how often `numStamps` disagreed with what you found.
3. The domain gap — source data isn't invoices. How did detections look on real invoices?
4. Per-class precision/recall/mean-IoU, and which class is weaker + your theory why.
5. Why category_id 2/3/4 (initials/redaction/date) were excluded.

Also note anything the next stage needs from you, and which figure you'd put on a slide.